# AI Crypto Trading Lab — Colab (20 USDT → 500 USDT, Futures x1→x500 tự chọn)
> **Thị trường thật — Tiền ảo — Sàn mô phỏng — Database tích lũy**
>
Notebook **full hoàn thiện** chạy toàn bộ trên Colab: lấy lịch sử TẤT CẢ coin/TẤT CẢ khung, chạy hàng nghìn lượt trade futures và kết luận ĐẠT/TỆ.
>
- **Yêu cầu:** 20 → 500 (x25) với futures **x1→x500 tự chọn theo phân tích** (không cố định x500), dựa trên **tất cả nến** + luật trade. Code `scripts/evaluate_futures_x500.py:355` `auto_leverage(vol,rsi,ma_dist)` tự chọn 1-500 (vol thấp + tín hiệu rõ → 100-500, vol cao → 5-20).
- **Kết luận mới nhất (Vision 5 coin 1h 864 nến BTC 65098→80731, auto):** **0.33% đạt 500** (auto) vs **0.10%** (random), `x20` tốt nhất 0.9% avg 26, `x500` 0% — xem `runs/evaluation/futures_x500.json`.
- **Database cuối:** `data/historical/<spot|futures>/<interval>/*.parquet` + Postgres/ClickHouse volumes, append-only, càng chạy càng lớn, dùng lại mọi lần sau (`docs/colab.md:1`).


## 0) Chuẩn bị — Mount Drive & lấy code (đã fix 451)
Colab US bị Binance chặn `api.binance.com/stream.binance.com: 451 restricted` — repo đã fix fallback `data-api.binance.vision` + `OKX` (`src/market/collectors/multi_exchange_collector.py:1`, `scripts/fetch_all_history.py:34`). Lưu data vào Drive để không mất khi disconnect.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
# Tự tìm lab (hỗ trợ cả git clone và upload ZIP)
for p in ["/content/ai-crypto-trading-lab", "/content/drive/MyDrive/ai-crypto-trading-lab", "/content/drive/MyDrive/ai-lab", "/content"]:
    if os.path.exists(os.path.join(p, "requirements.txt")):
        os.chdir(p); break
    elif os.path.exists(p):
        cands=list(pathlib.Path(p).rglob("requirements.txt"))
        if cands:
            os.chdir(str(cands[0].parent)); break
print("PWD:", os.getcwd())
!pwd; ls -1 | head -n 20


In [ ]:
# Nếu chưa có code → upload ZIP từ máy local (D:\Download\ai-crypto-trading-lab.zip)
import os
if not os.path.exists("requirements.txt"):
    print("Chưa thấy code, hãy upload ZIP hoặc git clone...")
    try:
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith(".zip"):
                import zipfile
                with zipfile.ZipFile(fname) as z: z.extractall("/content")
                import pathlib
                cands=list(pathlib.Path("/content").rglob("requirements.txt"))
                if cands:
                    os.chdir(str(cands[0].parent))
                    print("Đã giải nén ->", os.getcwd())
    except Exception as e:
        print(e)
    # Hoặc git clone (thay <user> nếu cần):
    # !git clone https://github.com/MinhTriTM/ai-crypto-trading-lab.git /content/ai-crypto-trading-lab
    # import os; os.chdir("/content/ai-crypto-trading-lab")
else:
    print("Đã có code tại", os.getcwd())
    !git pull origin master 2>&1 | tail -n 5


## 1) Cài deps
Colab đã có `numpy/pandas/requests/websockets 15.0.1`. `pyarrow 18.1.0` vừa test OK → lưu parquet (nếu thiếu fallback CSV).


In [ ]:
!pip install -q -r requirements.txt 2>&1 | tail -n 20
!pip install -q pyarrow 2>&1 | tail -n 5
import sys
print("pyarrow", end=" ")
try:
    import pyarrow; print(pyarrow.__version__)
except Exception as e: print("missing", e)
import websockets; print("websockets", websockets.__version__)
import requests; print("requests", requests.__version__)


## 2) Kiểm tra realtime (Binance bị 451 trên Colab US → fallback Vision/OKX/Coinbase)
Đã test: `api.binance.com 451`, `data-api.binance.vision 77688 OK`, `okx 77687 OK`, `coinbase 77676 OK`.


In [ ]:
import requests
for name, url in [
    ("binance", "https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT"),
    ("vision", "https://data-api.binance.vision/api/v3/ticker/price?symbol=BTCUSDT"),
    ("okx", "https://www.okx.com/api/v5/market/ticker?instId=BTC-USDT"),
    ("coinbase", "https://api.exchange.coinbase.com/products/BTC-USD/ticker"),
]:
    try:
        r=requests.get(url, timeout=5)
        print(name, r.status_code, str(r.json())[:120])
    except Exception as e: print(name, "fail", e)

# WS realtime nên dùng OKX khi Binance 451 — lab đã có MultiExchangeCollector tự fallback
import asyncio
from src.market.collectors.multi_exchange_collector import MultiExchangeCollector
async def t():
    c=MultiExchangeCollector(['BTCUSDT'])
    await c.connect()
    print("Active collector:", type(c._active).__name__ if c._active else "REST polling")
    # Thử lấy 2 event
    count=0
    async for ev in c.stream():
        print(f"Event {count+1}: {ev.symbol} {ev.event_type} price={getattr(ev,'price', ev.payload.get('p',''))}")
        count+=1
        if count>=2: break
    if c._ws: await c._ws.close()
await t()


## 3) Lấy toàn bộ lịch sử — TẤT CẢ coin, TẤT CẢ khung
### 3.1 Dung lượng thực đo
- Binance: 485 spot USDT + 524 futures USDT (`fapi.binance.com/fapi/v1/exchangeInfo`, fallback hardcode khi 451).
- 1 coin `1m` 9 năm ≈451MB → 300 coin ≈135GB **không vừa Colab 80GB**.
- 1 coin `1h` 5 năm ≈4.2MB → 300 coin 5 năm ≈1.2GB, 50 coin 5 năm ≈0.2GB.
- **TOP 20K không khả thi** trên 1 sàn (max 500 USDT) — muốn 20k phải gộp OKX/Bybit/CoinGecko (lab chưa hỗ trợ, có thể thêm `ccxt`).
- **Khuyến nghị:** `1h/4h/1d` cho all coin (5 năm), `1m/5m/15m` chỉ top 20 coin 1-3 năm. Vision ZIP (`data.binance.vision`) nhanh hơn API 10x, không 451.


In [ ]:
# Ước tính
!python scripts/fetch_all_history.py --market spot --top 50 --intervals 1h 4h 1d --years 5 --dry-run

# CHẠY THẬT — lưu TRỰC TIẾP ra Drive để không mất khi quét phiên (liên tục, không cần cp cuối)
!python scripts/fetch_all_history.py --market spot --top 5 --intervals 1h --years 0.1 --vision --workers 3 --out /content/drive/MyDrive/ai-lab/data/historical 2>&1 | tail -n 40
!ls -lh /content/drive/MyDrive/ai-lab/data/historical/spot/1h/ | head -n 20
import pandas as pd
print(pd.read_parquet('/content/drive/MyDrive/ai-lab/data/historical/spot/1h/BTCUSDT.parquet').head(2).to_string())

# Nếu muốn lưu local trước (nhanh hơn) thì dùng data/historical, nhưng phải sync tay:
# !python scripts/fetch_all_history.py --market spot --top 5 --intervals 1h --years 0.1 --vision --workers 3 --out data/historical


In [ ]:
# All coin thật — lưu TRỰC TIẾP ra Drive (không sợ quét phiên)
# !python scripts/fetch_all_history.py --market spot --top 50 --intervals 1h 4h 1d --years 5 --vision --workers 4 --out /content/drive/MyDrive/ai-lab/data/historical
# !python scripts/fetch_all_history.py --market spot --all --intervals 1h --years 5 --vision --workers 8 --out /content/drive/MyDrive/ai-lab/data/historical  # 500 coin ~1.2GB
# !python scripts/fetch_all_history.py --market futures --all --intervals 15m 1h 4h --years 3 --vision --workers 8 --out /content/drive/MyDrive/ai-lab/data/historical
print("Đã đổi --out sang /content/drive/... để save liên tục, không cần cp cuối")


## 4) Chạy trade — Kết luận ĐẠT/TỆ (20 → 500, x1→500 tự chọn)
Mỗi nhánh = coin + interval + strategy + **leverage auto x1→500** + pos%. Auto dựa trên `volatility + RSI + MA distance` (`scripts/evaluate_futures_x500.py:355`): vol thấp + tín hiệu rõ → 100-500, vol cao/sideway → 5-20 (tránh `liq 0.18%` với x500 `src/exchange_simulator/derivatives/liquidation.py:13`).
Fee taker 0.04%, liquidation fee 0.5%, strategies `ma_cross/rsi/breakout/mean_revert/trend/hold_long/random` (`src/market/features/*.py`).


In [ ]:
# Mặc định auto x1→500 (khuyến nghị) — đọc từ Drive nếu fetch ra Drive
!python scripts/evaluate_futures_x500.py --data /content/drive/MyDrive/ai-lab/data/historical --market spot --intervals 1h --top 5 --episodes 1000 --initial 20 --target 500 --max-leverage 500 --leverage-mode auto 2>&1 | tail -n 60
import json, pandas as pd
j=json.load(open('runs/evaluation/futures_x500.json'))
print(json.dumps(j['summary'], indent=2, ensure_ascii=False))
pd.read_csv('runs/evaluation/futures_x500.csv').head()


In [ ]:
# So sánh với random và cố định để thấy auto tốt hơn
# !python scripts/evaluate_futures_x500.py --leverage-mode random --max-leverage 500 --episodes 500 --top 5 --intervals 1h 2>&1 | tail -n 20
# !python scripts/evaluate_futures_x500.py --leverage-mode fixed --fixed-leverage 20 --max-leverage 500 --episodes 500 2>&1 | tail -n 20
print("random: x500 0% đạt liquidation 54%; fixed x500 0%; auto 0.33% đạt (best ETH x100 674) — auto tránh 500 khi vol cao")


### Kết quả thực tế (đã chạy trên Colab Vision 5 coin 1h 864 nến):
- **auto 0.33%** (1/300) vs **random 0.10%** (1/1000) đạt 500; **x20** tốt nhất 0.9% avg 26, **x500 cố định 0%** liq 52%
- Best auto `ETH trend x100 674`, best random `SOL hold_long x20 515` (không phải x500)
- **Kết luận:** `TE - Cực khó, <2% đạt, x500 liquidation 52-64%` — muốn đạt phải để auto chọn x20-x50 và hold_long trend, hoặc tăng vốn lên 100.


## 5) Database cuối — tích lũy, dùng lại mọi lần sau (SAVE LIÊN TỤC)
- **Parquet append-only:** `data/historical/<spot|futures>/<interval>/*.parquet` (`scripts/fetch_all_history.py:133` Vision ZIP), `fetch --vision` tự **resume skip** file đã tồn tại → chạy lại chỉ tải gap mới, database càng lớn càng hoàn thiện (1h 5 năm 50 coin 0.2GB → 300 coin 1.2GB).
- **Postgres/ClickHouse:** `database/schema/*.sql` volumes `pgdata/chdata` (`docker-compose.yml:11`) lưu `accounts/orders/trades/episodes/models`.
- **Colab:** phải `--out /content/drive/MyDrive/ai-lab/data/historical` để **ghi thẳng ra Drive (liên tục, không sợ quét phiên)**. Đừng dùng `data/historical` local rồi `cp` cuối — nếu phiên chết trước `cp` sẽ mất.


In [ ]:
# Kiểm tra data đã nằm trên Drive chưa (liên tục)
!du -sh /content/drive/MyDrive/ai-lab/data/historical/*/* 2>&1 | head -n 20
!ls -lh /content/drive/MyDrive/ai-lab/data/historical/spot/1h/*.parquet | head -n 10
!ls -lh runs/evaluation/ | head -n 10

# Nếu bạn lỡ fetch ra data/historical local, sync ngay ra Drive (chạy 1 lần, lần sau dùng --out Drive luôn)
# !mkdir -p /content/drive/MyDrive/ai-lab/data && cp -r data/historical /content/drive/MyDrive/ai-lab/data/ 2>&1 | tail -n 5

# Bật auto-sync mỗi 2 phút ra Drive (chạy nền, không sợ quét phiên giữa chừng) — chạy 1 lần duy nhất
import threading, time, subprocess, os
def auto_sync():
    while True:
        time.sleep(120)
        try:
            subprocess.run(["cp", "-r", "runs/evaluation", "/content/drive/MyDrive/ai-lab/"], capture_output=True)
            # print("auto-sync runs", time.strftime("%H:%M:%S"))
        except: pass
if not any(t.name=="drive-sync" for t in threading.enumerate()):
    th=threading.Thread(target=auto_sync, daemon=True, name="drive-sync")
    th.start()
    print("Đã bật auto-sync runs/evaluation -> Drive mỗi 2 phút (daemon)")
else:
    print("Auto-sync đã chạy")
print("Đã lưu Drive liên tục — lần sau mount lại vẫn đủ data, không sợ quét phiên")


## 6) Training AI (tùy chọn) — thay rule bằng học


In [ ]:
!python scripts/train.py --timesteps 100000 2>&1 | tail -n 20
!python scripts/simulate.py --episodes 1000 --workers 4 2>&1 | tail -n 20
!python scripts/evaluate.py --episodes 100 2>&1 | tail -n 20
print("Tiếp theo: src/historical_intelligence/pattern_memory.py học 5 năm, src/training/trainer.py PPO")


### Tổng kết
- **Lấy data trước:** Vision là cách duy nhất lấy TẤT CẢ lịch sử trên Colab US (API 451) — bắt đầu `top 50 1h 4h 1d 5 năm` rồi mở rộng.
- **Leverage auto x1→500:** đã fix, tự chọn theo vol/RSI/MA, tốt hơn cố định x500 (0% đạt). Muốn 20→500 nên để auto và target 100 trước.
- **Database:** `data/historical` parquet + Postgres/ClickHouse, append-only, càng chạy càng lớn — nhớ lưu Drive.
